In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime
import numpy as np

In [2]:
def scrape_autoscout(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find all car listings on the page
    cars = soup.find_all('article', class_='cldt-summary-full-item')

    # Prepare a list to store the car data
    car_data = []

    for car in cars:
        # Extract the data from each car listing
        url = car.find('a', class_="ListItem_title__ndA4s")['href']
        url = 'https://www.autoscout24.es' + url
        location = car.find('span', class_="SellerInfo_address__leRMu")
        if location is None:
            location = car.find('span', class_="SellerInfo_private__THzvQ").text
        else:
            location = location.text

        title_element = car.find('h2')
        description = title_element.find('span', class_='ListItem_version__5EWfi').text.strip()
        title = title_element.text.replace(description, '').strip()
        try:
            brand = title.split()[0]
        except IndexError:
            brand = np.nan

        price = car.find('p', class_='Price_price__APlgs').text.strip()
        price = int(re.sub(r'\D', '', price))

        details_elements = car.find_all('span', class_='VehicleDetailTable_item__4n35N')
        details = [detail.text.strip() for detail in details_elements]
        mileage = details[0]
        try:
            mileage = int(re.sub(r'\D', '', mileage))
        except ValueError:
            mileage = np.nan
        transmission = details[1]
        year = details[2]
        # year = datetime.strptime(year, '%m/%Y')
        engine_type = details[3]
        try:
            cv = details[4]
            cv = int(re.search(r'(\d+) CV', cv).group(1))
        except AttributeError:
            cv = np.nan

        # Add the data to our list
        car_data.append([brand, title, description, price, mileage, transmission, year, engine_type, cv, location, url])

    # Convert the list to a DataFrame and return it
    return pd.DataFrame(car_data, columns=['brand', 'title', 'description', 'full_price', 'mileage', 'transmission', 'year', 'fuel', 'cv', 'location', 'url']
)

def scrape_multiple_pages(base_url):
    response = requests.get(base_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find max page 
    page = soup.find('li', class_='pagination-item--disabled pagination-item--page-indicator')
               
    if page is None:
        return
    else:
        page= int(page.text.split()[-1])

    # Prepare a list to store the DataFrames
    dfs = []

    for i in range(1, page + 1):
        # Construct the URL for the current page
        url = base_url + '&page=' + str(i)

        # Scrape the current page and store the DataFrame
        df = scrape_autoscout(url)
        dfs.append(df)

    # Concatenate all the DataFrames
    master_df = pd.concat(dfs, ignore_index=True)

    return master_df


In [4]:
# df.to_csv(f'data/autoscout/{datetime.now().strftime("%y.%m.%d")}_autoscout_raw.csv', index=False)

In [3]:
df = pd.DataFrame()
for i in range(240):
    # url = f'https://www.autoscout24.es/lst?damaged_listing=exclude&desc=0&lat=40.41669&lon=-3.70035&powertype=kw&pricefrom={(i+20)*3}00&priceto={(i+21)*3}00&sort=price&zip=madrid&zipr=20'
    url = f'https://www.autoscout24.es/lst?atype=C&cy=E&damaged_listing=exclude&desc=0&fregfrom=2016&fregto=2023&powerfrom=74&powerto=118&powertype=hp&pricefrom={(i+10)*2}00&priceto={(i+11)*2}00&sort=price'
    df = pd.concat([df, scrape_multiple_pages(url)],ignore_index=True)
    df = df.drop_duplicates()
    print(df.shape)
    
df.to_csv(f'data/autoscout/{datetime.now().strftime("%y.%m.%d")}_autoscout_raw.csv', index=False)

(0, 0)
(0, 0)
(0, 0)
(0, 0)
(0, 0)
(0, 0)
(1, 11)
(2, 11)
(2, 11)
(2, 11)
(3, 11)
(3, 11)
(3, 11)
(3, 11)
(4, 11)
(4, 11)
(4, 11)
(5, 11)
(8, 11)
(14, 11)
(15, 11)
(22, 11)
(43, 11)
(46, 11)
(106, 11)
(111, 11)
(143, 11)
(187, 11)
(196, 11)
(316, 11)
(325, 11)
(352, 11)
(587, 11)
(648, 11)
(1045, 11)
(1113, 11)
(1195, 11)
(1571, 11)
(1749, 11)
(2132, 11)
(2283, 11)
(2480, 11)
(2870, 11)
(3270, 11)
(3642, 11)
(3943, 11)
(4333, 11)
(4713, 11)
(5113, 11)
(5513, 11)
(5913, 11)
(6313, 11)
(6713, 11)
(7113, 11)
(7513, 11)
(7913, 11)
(8313, 11)
(8713, 11)
(9113, 11)
(9513, 11)
(9913, 11)
(10313, 11)
(10713, 11)
(11113, 11)
(11513, 11)
(11913, 11)
(12313, 11)
(12713, 11)
(13113, 11)
(13513, 11)
(13913, 11)
(14313, 11)
(14713, 11)
(15113, 11)
(15513, 11)
(15913, 11)
(16313, 11)
(16713, 11)
(17113, 11)
(17513, 11)
(17913, 11)
(18313, 11)
(18713, 11)
(19113, 11)
(19513, 11)
(19913, 11)
(20313, 11)
(20713, 11)
(21113, 11)
(21513, 11)
(21913, 11)
(22313, 11)
(22713, 11)
(23113, 11)
(23513, 11)
(239

KeyboardInterrupt: 

In [ ]:
# df = pd.DataFrame()
# for i in range(70):
#     url = f'https://www.autoscout24.es/lst?atype=C&body=4%2C5%2C12&cy=E&damaged_listing=exclude&desc=0&fregfrom=2017&kmfrom=20000&kmto=100000&lat=40.41669&lon=-3.70035&powerfrom=59&powertype=hp&pricefrom={(i+30)*2}00&priceto={(i+31)*2}00&search_id=49cs04bi9u&sort=price&source=homepage_search-mask&ustate=N%2CU&zip=madrid-%28comunidad-de-madrid%29&zipr=50'
#     df = pd.concat([df, scrape_multiple_pages(url)],ignore_index=True)
#     df = df.drop_duplicates()
#     print(df.shape)
    
# df.to_csv(f'data/autoscout/{datetime.now().strftime("%y.%m.%d")}_autoscout_raw.csv', index=False)